# カスタムミドルウェア-Wrap-style hooks
## 1、wrap_model_call の使用

### 1.1 デコレーターに基づく実装

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# .env ファイルから環境変数を読み込む
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE")
)

In [2]:

from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware


@wrap_model_call
def wrap_model_call_middleware(request: ModelRequest,
                               handler: Callable[[ModelRequest], ModelResponse], ) -> ModelResponse | None:
    request.messages[-1].content += "---> wrap_model_call_before <---"

    # モデルの呼び出し
    response = handler(request)

    response.result[0].content += "---> wrap_model_call_after <---"

    return response

In [3]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

こんにちは---> wrap_model_call_before <---
================================== Ai Message ==================================

こんにちは！どうされましたか？---> wrap_model_call_after <---


### 1.2 クラスに基づく実装

In [4]:
from langchain.agents.middleware import AgentMiddleware


class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest,
                        handler: Callable[[ModelRequest], ModelResponse], ) -> ModelResponse | None:
        request.messages[-1].content += "---> wrap_model_call_before <---"

        # モデルの呼び出し
        response = handler(request)

        response.result[0].content += "---> wrap_model_call_after <---"

        return response


agent = create_agent(
    model=model,
    middleware=[
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

こんにちは---> wrap_model_call_before <---
================================== Ai Message ==================================

こんにちは！  
`wrap_model_call_before` は、たぶん「モデル呼び出しの前に何かを差し込む」ためのフック名っぽいですね。  

もし意図があれば、たとえば：

- 呼び出し前にログを出す
- 入力を整形する
- 権限チェックをする
- プロンプトに前処理を入れる

みたいな用途で使えます。  
必要なら、Python / JavaScript / LangChain / 独自実装のどれかに合わせて具体例を書けます。---> wrap_model_call_after <---


## 2、wrap_tool_call の使用

### 2.1 デコレーターに基づく実装

In [5]:

from langchain_core.tools import tool
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call


@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    当日の指定都市の天気を取得する

    Args:
        city: 都市名
        is_forcast: 明日の天気予報を含めるかどうか
    """
    res = f"{city}は今日良い天気です"
    if is_forcast:
        res += "\n明日も良い天気です"
    return res


@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
                              handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                              ) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"元のパラメータ：{request.tool_call['args']}")
    print(f"元のパラメータでの呼び出し結果：{result}")

    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)

    print(f"更新後のパラメータ：{request.tool_call['args']}")
    print(f"更新後のパラメータでの呼び出し結果：{result}")
    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages": [HumanMessage("北京の今日の天気を調べてください")]
})

for msg in response["messages"]:
    msg.pretty_print()

元のパラメータ：{'city': '北京', 'is_forcast': False}
元のパラメータでの呼び出し結果：content='北京は今日良い天気です' name='get_weather' tool_call_id='call_si8fjO1Vzrh5kr7umVsOQAwb'
更新後のパラメータ：{'city': '北京', 'is_forcast': True}
更新後のパラメータでの呼び出し結果：content='北京は今日良い天気です\n明日も良い天気です' name='get_weather' tool_call_id='call_si8fjO1Vzrh5kr7umVsOQAwb'
================================ Human Message =================================

北京の今日の天気を調べてください
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_si8fjO1Vzrh5kr7umVsOQAwb)
 Call ID: call_si8fjO1Vzrh5kr7umVsOQAwb
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京は今日良い天気です
明日も良い天気です
================================== Ai Message ==================================

北京の今日の天気は**良い天気**です。  
**明日も良い天気**のようです。


### 2.2 クラスに基づく実装

In [6]:

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self, request: ToolCallRequest,
                       handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                       ) -> ToolMessage | Command[Any]:
        result = handler(request)
        print(f"元のパラメータ：{request.tool_call['args']}")
        print(f"元のパラメータでの呼び出し結果：{result}")

        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)

        print(f"更新後のパラメータ：{request.tool_call['args']}")
        print(f"更新後のパラメータでの呼び出し結果：{result}")
        return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[WrapToolCallMiddleware()]
)

response = agent.invoke({
    "messages": [HumanMessage("上海の今日の天気を調べてください")]
})

for msg in response["messages"]:
    msg.pretty_print()

元のパラメータ：{'city': '上海', 'is_forcast': False}
元のパラメータでの呼び出し結果：content='上海は今日良い天気です' name='get_weather' tool_call_id='call_7S2Htx9WzO6ZoFwSjEb4Je7B'
更新後のパラメータ：{'city': '上海', 'is_forcast': True}
更新後のパラメータでの呼び出し結果：content='上海は今日良い天気です\n明日も良い天気です' name='get_weather' tool_call_id='call_7S2Htx9WzO6ZoFwSjEb4Je7B'
================================ Human Message =================================

上海の今日の天気を調べてください
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_7S2Htx9WzO6ZoFwSjEb4Je7B)
 Call ID: call_7S2Htx9WzO6ZoFwSjEb4Je7B
  Args:
    city: 上海
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

上海は今日良い天気です
明日も良い天気です
================================== Ai Message ==================================

上海の今日の天気は**良い天気**です。  
**明日も良い天気**の予報です。
